# Sesión 4 · RAG: recuperar, medir y responder con evidencia
**Databricks AI Engineer** — caso Neptuno

En este laboratorio construiremos un RAG pequeño pero completo:

1. conservamos el texto crudo y normalizamos sin perder offsets;
2. separamos span citable, texto de embedding y contexto de respuesta;
3. comparamos BM25, dense y fusión RRF;
4. generamos una respuesta con cita y una abstención;
5. guardamos un gold set para no ajustar por intuición.

### Intuición rápida: relaciones en el espacio vectorial

Ejemplo clásico: si restas el vector de **Hombre** a **Rey** y le sumas **Mujer**, el
resultado matemático estará muy cerca del vector de **Reina**:

`Rey - Hombre + Mujer ≈ Reina`

Es una analogía ilustrativa de embeddings de palabras: una relación puede aparecer como
una dirección en el espacio vectorial. No es una garantía exacta de todos los modelos
modernos ni sustituye la evaluación sobre nuestro corpus.

### Antes de empezar · catálogo del curso

Al importar o actualizar el notebook, Databricks no ejecuta el código automáticamente.
Esta debe ser la primera celda que corras. Escribe arriba el nombre completo del catálogo
de S01 y S02, por ejemplo: `neptuno_tunombre`.

In [0]:
dbutils.widgets.text("catalogo", "", "Tu catálogo de S01–S02")
print("✅ Widget creado. Escribe arriba el nombre completo de tu catálogo y vuelve a ejecutar esta celda.")

✅ Widget creado. Escribe arriba el nombre completo de tu catálogo y vuelve a ejecutar esta celda.


## 0 · Configuración

Cada alumno usa su catálogo `neptuno_<nombre>`. La validación docente puede usar otro schema
para no tocar las tablas de clase.

In [0]:
import hashlib
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict

from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

dbutils.widgets.text("schema", "rag", "Schema del laboratorio")
#dbutils.widgets.text("modelo", "system.ai.gpt-5-6-luna", "Modelo generador") en free edition no disponible
dbutils.widgets.text("modelo", "databricks-gpt-oss-120b", "Modelo generador")
dbutils.widgets.text("embedding_endpoint", "databricks-qwen3-embedding-0-6b", "Endpoint de embeddings")
dbutils.widgets.text("top_k", "4", "Candidatos por carril")

CATALOGO = dbutils.widgets.get("catalogo").strip().lower()
SCHEMA = dbutils.widgets.get("schema").strip().lower()
MODELO = dbutils.widgets.get("modelo").strip()
EMBEDDING_ENDPOINT = dbutils.widgets.get("embedding_endpoint").strip()
TOP_K = int(dbutils.widgets.get("top_k"))

assert re.fullmatch(r"neptuno_[a-z0-9_]+", CATALOGO or ""), "Usa neptuno_<nombre>."
assert re.fullmatch(r"rag(?:_[a-z0-9_]+)?", SCHEMA or ""), "Usa rag o rag_<sufijo>."
assert re.fullmatch(r"[a-z0-9_.-]+", EMBEDDING_ENDPOINT), "Endpoint inválido."
assert 1 <= TOP_K <= 10
assert CATALOGO in {r.catalog.lower() for r in spark.sql("SHOW CATALOGS").collect()}

NS = f"{CATALOGO}.{SCHEMA}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NS}")
print(f"✅ namespace: {NS}")

✅ namespace: neptuno_emerson_suarez.rag


### ⭐ Checkpoint · endpoint de embeddings real

Para español usamos `databricks-qwen3-embedding-0-6b`: es multilingüe, acepta lotes y
devuelve hasta 1.024 dimensiones. La disponibilidad depende de región y workspace, por eso
validamos primero y fallamos con un mensaje accionable.
La celda anterior lista los endpoints visibles: úsala para confirmar el nombre real antes de
cambiar el widget de embeddings.

In [0]:
w = WorkspaceClient()
endpoints_activos = [ep.name for ep in w.serving_endpoints.list()]
nombre_widget = EMBEDDING_ENDPOINT
if nombre_widget in endpoints_activos:
    print(f"✅ El endpoint '{nombre_widget}' sí existe.")
else:
    print(f"❌ El endpoint '{nombre_widget}' NO existe en este espacio de trabajo.")
    print("Endpoints disponibles:", endpoints_activos)
endpoints = {ep.name: ep for ep in w.serving_endpoints.list()}
assert EMBEDDING_ENDPOINT in endpoints, (
    f"No existe {EMBEDDING_ENDPOINT}. Revisa la lista anterior y actualiza el widget."
)

estado = str(endpoints[EMBEDDING_ENDPOINT].state.ready)
assert "READY" in estado.upper(), f"El endpoint existe, pero su estado es {estado}."
print(f"✅ {EMBEDDING_ENDPOINT}: READY")

✅ El endpoint 'databricks-qwen3-embedding-0-6b' sí existe.
✅ databricks-qwen3-embedding-0-6b: READY


In [0]:
probe_textos = [
    "devolución de productos refrigerados",
    "política de cadena de frío",
    "pronóstico del clima",
]
probe = (
    spark.createDataFrame([(x,) for x in probe_textos], "texto string")
    .withColumn("embedding", F.expr(f"ai_query('{EMBEDDING_ENDPOINT}', texto)"))
    .collect()
)

def cosine(a, b):
    numerador = sum(x * y for x, y in zip(a, b))
    norma = math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b))
    return numerador / norma if norma else 0.0

dims = {len(r.embedding) for r in probe}
sim_relacionada = cosine(probe[0].embedding, probe[1].embedding)
sim_ajena = cosine(probe[0].embedding, probe[2].embedding)
assert dims == {1024}, f"Dimensión inesperada: {dims}"
assert sim_relacionada > sim_ajena, "El smoke test semántico no separó el par relacionado."
print(f"✅ 3 vectores · 1.024 dims · relacionada={sim_relacionada:.3f} · ajena={sim_ajena:.3f}")

✅ 3 vectores · 1.024 dims · relacionada=0.728 · ajena=0.545


### ⭐ Experimento: una relación como dirección vectorial

Ahora lo comprobamos con el endpoint real. Calculamos `Rey - Hombre + Mujer` y medimos
qué tan cerca queda del embedding de `Reina`. El porcentaje es una forma didáctica de
mostrar la similitud coseno; no es una probabilidad ni una garantía del modelo.

In [0]:
palabras_analogia = ["Rey", "Hombre", "Mujer", "Reina"]
df_analogia = spark.createDataFrame([(x,) for x in palabras_analogia], "palabra string")
filas_analogia = (
    df_analogia
    .withColumn("embedding", F.expr(f"ai_query('{EMBEDDING_ENDPOINT}', palabra)"))
    .collect()
)
vectores_analogia = {fila.palabra: fila.embedding for fila in filas_analogia}
assert {len(vector) for vector in vectores_analogia.values()} == {1024}

tabla_vectores = spark.createDataFrame(
    [(palabra, vectores_analogia[palabra]) for palabra in palabras_analogia],
    "palabra string, vector array<double>",
)
display(tabla_vectores)
print("✅ Tabla de palabras y vectores: 4 filas · 1.024 dimensiones por vector")

vector_resultado = [
    rey - hombre + mujer
    for rey, hombre, mujer in zip(
        vectores_analogia["Rey"],
        vectores_analogia["Hombre"],
        vectores_analogia["Mujer"],
    )
]
display(
    spark.createDataFrame(
        [("Rey - Hombre + Mujer", vector_resultado)],
        "operacion string, vector_resultante array<double>",
    )
)

similitud_reina = cosine(vector_resultado, vectores_analogia["Reina"])
parecido_pct = max(0.0, min(1.0, similitud_reina)) * 100

display(
    spark.createDataFrame(
        [("Rey - Hombre + Mujer", "Reina", float(similitud_reina), float(parecido_pct))],
        "vector_resultante string, vector_referencia string, similitud_coseno double, parecido_porcentaje double",
    )
)
print(f"✅ Comparación contra Reina · parecido orientativo={parecido_pct:.2f}%")

palabra vector Rey List(0.02349032834172249, 0.013158115558326244, -0.012804877944290638, -0.033204372972249985, 0.0370899923145771, -0.02808242291212082, -0.021900756284594536, 0.009625735692679882, -0.024726660922169685, -0.01121530681848526, -0.02790580317378044, 0.01854499615728855, 0.0734735056757927, -0.012716568075120449, -0.05157275125384331, 0.11162321269512177, -0.007373843342065811, 0.10738435387611389, 0.08830950409173965, -0.007859545759856701, 0.016425566747784615, -0.007197224535048008, 0.007594617083668709, 0.0967872142791748, 0.0540454164147377, 0.057577796280384064, -0.10102607309818268, 0.10314550250768661, 0.03249789774417877, -0.016248948872089386, -0.01863330602645874, 0.025786375626921654, 0.05899074673652649, -0.01218671165406704, 0.0057842726819217205, -0.016072329133749008, -0.019074853509664536, -0.025079898536205292, -0.047333892434835434, 0.05263246223330498, 0.058284271508455276, 0.0039297728799283504, -0.016248948872089386, 0.003002523211762309, -0.02631623111665249, -0.017220353707671165, 0.042388562113046646, -0.012451640330255032, 0.04556770250201225, -0.00152333895675838, -0.03726661205291748, -0.010552985593676567, 0.020841043442487717, 0.035147182643413544, -0.0015343775739893317, -0.022077376022934914, 0.05121951177716255, -0.04945332184433937, 0.059343986213207245, -0.0021083892788738012, -0.010950378142297268, 0.06287636607885361, -0.046274181455373764, -0.03479394316673279, 0.018280066549777985, 0.0653490349650383, 0.024903280660510063, -0.0017441126983612776, -0.04592094197869301, -0.003731076605618, 0.010994533076882362, -0.03567703813314438, -0.025786375626921654, 0.041505467146635056, 0.025433136150240898, 0.036030277609825134, -6.3748424872756E-4, -0.11303616315126419, 0.018898233771324158, 0.003532380098477006, 0.011480235494673252, 0.06640874594449997, 0.009978974238038063, 0.02808242291212082, -0.012893187813460827, 0.05157275125384331, 0.04168208688497543, -0.021724138408899307, -0.016425566747784615, -0.007506307680159807, 0.0063582840375602245, 0.025962993502616882, 0.007462153211236, -0.02437342330813408, 0.04132884740829468, -0.021812446415424347, -1.4143319276627153E-4, -0.002075273310765624, -0.02243061363697052, -0.004945332184433937, -0.014217830263078213, 0.010155593045055866, 0.034970562905073166, 0.001385355368256569, -0.025609755888581276, 0.08124474436044693, -0.061110176146030426, 0.020222876220941544, -0.03638351708650589, -0.058284271508455276, 0.0071530696004629135, 0.0169554241001606, -0.014482758939266205, 0.00609335582703352, -0.01545416284352541, 0.0039297728799283504, -0.07771236449480057, -0.05333894118666649, -0.019251471385359764, -0.0033336838241666555, 0.03903280198574066, -0.0028148654382675886, -0.03797308728098869, -0.02790580317378044, -0.01545416284352541, 0.03920941799879074, -0.028435660526156425, -0.002185660181567073, 0.026492850854992867, -0.06040370091795921, -0.021282590925693512, -0.06888141483068466, 0.009669890627264977, -0.006932295858860016, 0.0411522276699543, -0.006579057779163122, 0.012628259137272835, -4.1119111119769514E-4, 0.04415475204586983, -0.002980445744469762, -0.01880992390215397, 8.610176737420261E-4, -0.004768713377416134, -0.05227922648191452, 0.007241379469633102, 0.018368376418948174, -0.011568545363843441, -0.04291841760277748, 0.026492850854992867, -0.011656854301691055, 0.03656013309955597, 0.004680403508245945, -0.023137090727686882, -0.027375945821404457, -0.043448276817798615, 0.020752733573317528, -0.02790580317378044, -0.03444070741534233, 0.016160639002919197, 0.01766190119087696, 0.01218671165406704, 0.016425566747784615, -0.020222876220941544, -0.025962993502616882, -0.027729183435440063, 0.005872582085430622, -0.027552565559744835, -0.022783851251006126, -0.019957948476076126, 0.013599663972854614, -0.01598401926457882, 0.04662741720676422, -0.009890664368867874, 0.00936080701649189, 0.01545416284352541, -0.034087467938661575, -0.022960470989346504, -0.08159798383712769, -0.012451640330255032, 0.017308663

✅ Tabla de palabras y vectores: 4 filas · 1.024 dimensiones por vector


operacion vector_resultante Rey - Hombre + Mujer List(0.0342669771052897, 0.09140163380652666, -0.0118112126365304, -0.05048147263005376, 0.072610754519701, 0.0067201415076851845, 0.027844803407788277, 0.0536223363596946, 0.03988645784556866, -0.057215576991438866, -0.02619992569088936, 0.043105210177600384, 0.10947973281145096, -0.012316595762968063, -0.07408730313181877, 0.03008596319705248, -0.02103670034557581, 0.12413868308067322, 0.13426577299833298, -0.008240320719778538, 0.040495953522622585, -0.06250118487514555, 7.363054901361465E-4, 0.1305457055568695, 0.056704115122556686, 0.04786272719502449, -0.11779338121414185, 0.04373659938573837, 0.07306808419525623, 0.022177021950483322, 0.008700616657733917, -0.0023823524825274944, 0.04757143743336201, -0.03156706504523754, -0.0054029920138418674, -0.013532974757254124, -0.01627460855524987, -0.04236991493962705, -0.05376442149281502, 0.08842538855969906, 0.09102711174637079, 0.05686054611578584, 0.017751088831573725, 0.004924637032672763, -0.010190834291279316, -0.0655683137010783, 0.09435480460524559, 0.03711700113490224, 0.09597979777026922, -0.008955327386502177, -0.06558892596513033, 0.010942010208964348, -0.008233323693275452, 0.04284185916185379, -0.005513876094482839, -0.03806287236511707, 0.05028656870126724, -0.032776255160570145, 0.028121743351221085, -0.02412929548881948, -0.0017689010128378868, 0.03536667674779892, -0.035771965980529785, 0.023966118693351746, 0.03652047552168369, 0.04923257231712341, 0.029590891674160957, 0.017240545945242047, -0.08704044297337532, -0.005486514884978533, -0.005538637284189463, -0.05478896014392376, -0.05263373861089349, 0.06212829519063234, 0.01782949361950159, 0.010829895734786987, -0.01645556604489684, -0.10986339300870895, 0.006713850423693657, 0.014254361391067505, 0.023618237813934684, 0.03216708451509476, -0.022636139765381813, 0.037147413939237595, -0.031035478226840496, 0.008182995021343231, 0.03672410920262337, 0.013331450521945953, 0.008243996649980545, -0.01286584185436368, 0.00832992559298873, 0.03975015878677368, 0.016751252580434084, 0.01526562124490738, 0.04553234949707985, -0.04355798754841089, 0.0027214854635531083, -0.009187493240460753, -0.016726097092032433, -0.025511269690468907, -0.020197752863168716, -0.006172312423586845, -0.011380638927221298, -0.0038540754467248917, -0.028632186353206635, 0.109786756336689, -0.060017041862010956, 0.012711567338556051, -0.045196544378995895, -0.061645541340112686, 0.014370039105415344, 0.026476800441741943, -7.769092917442322E-6, 0.07178211491554976, 0.01667524967342615, -0.015884498599916697, -0.05875684041529894, -0.05496307462453842, -0.02516787312924862, 0.01776570244692266, 0.008680160623043776, -0.03462897124700248, -0.042808445170521736, -0.0020740237087011337, 0.02985155861824751, 0.04180966876447201, -0.0293086227029562, 0.0045586032792925835, 0.03446871601045132, -0.06394621729850769, -0.043014155875425786, -0.06151540204882622, -0.004818228771910071, 0.02202818263322115, 0.04546021483838558, 5.211529787629843E-4, -0.026865161024034023, 0.02594009824679233, 0.08410773985087872, -0.007284805993549526, -0.028155481442809105, -0.02014967711875215, -0.013575931312516332, -0.05391108617186546, 0.011659423355013132, 0.005058138747699559, -0.007872770307585597, -0.05839673156151548, 0.012962250038981438, 0.012375019490718842, 0.08436745963990688, 0.00903040706180036, -0.04857298266142607, -0.0187288885936141, -0.0012512416578829288, -0.0012556519359350204, -0.04151635151356459, -0.024760380387306213, 0.03827604081016034, 0.022905046120285988, 0.017103802732890472, 0.01835820823907852, -0.011525991256348789, -0.04583352478221059, -0.04050389863550663, 0.0019703758880496025, -0.027778398245573044, -0.036844209767878056, -0.06497000902891159, -0.005041731987148523, -0.009556981476634974, 0.02903580665588379, 0.0024090316146612167, -0.007142284302972257, 0.0012732481118291616, -0.061954984441399574, -0.037112779915332794, -0.011627886444330215, -0.03497086698189378, 0.01

vector_resultante,vector_referencia,similitud_coseno,parecido_porcentaje
Rey - Hombre + Mujer,Reina,0.7880491227571881,78.8049122757188


✅ Comparación contra Reina · parecido orientativo=78.80%


## 1 · Corpus crudo de Neptuno

El primer documento incluye una palabra cortada por guion y salto de línea. Es deliberado:
nos permite comprobar que la normalización mejora búsqueda sin reescribir la evidencia original.

In [0]:
documentos = [
    (
        "politica_devoluciones",
        "Política de devoluciones",
        "2026-07-01",
        "Las devoluciones de alimentos no pere-\ncibles se aceptan hasta 30 días después de la entrega. "
        "Productos refrigerados o congelados solo se aceptan dentro de 24 horas si existe evidencia "
        "de ruptura de cadena de frío. El cliente debe informar el número de pedido.",
    ),
    (
        "contrato_expreso_veloz",
        "Contrato Expreso Veloz",
        "2026-06-15",
        "Expreso Veloz cubre Lima Metropolitana. El compromiso estándar es entregar en 24 horas hábiles. "
        "Si supera 48 horas, Neptuno puede solicitar una nota de crédito equivalente al 8 por ciento del flete.",
    ),
    (
        "contrato_paquetes_unidos",
        "Contrato Paquetes Unidos",
        "2026-06-15",
        "Paquetes Unidos cubre provincias. El plazo objetivo es de 3 a 5 días hábiles. Mercadería refrigerada "
        "requiere embalaje térmico provisto por Neptuno; el transportista registra temperatura en recepción y entrega.",
    ),
    (
        "ficha_bebidas",
        "Ficha de categoría Bebidas",
        "2026-05-20",
        "La categoría Bebidas incluye cafés, tés, cervezas y vinos importados. Los vinos requieren almacenamiento "
        "sin luz directa entre 12 y 18 grados Celsius. No deben colocarse junto a productos con olores intensos.",
    ),
    (
        "ficha_lacteos",
        "Ficha de categoría Lácteos",
        "2026-05-20",
        "Los productos lácteos se reciben entre 2 y 6 grados Celsius. Una lectura fuera de rango debe registrarse "
        "como incidencia y el lote queda en cuarentena hasta evaluación de calidad.",
    ),
    (
        "proveedor_exoticos",
        "Ficha del proveedor Exóticos Líquidos",
        "2026-04-10",
        "Exóticos Líquidos provee principalmente Bebidas desde Reino Unido. El pedido mínimo es de 20 cajas y "
        "el lead time habitual es 18 días. Contacto operativo: compras internacionales.",
    ),
]

df_docs = spark.createDataFrame(
    documentos,
    "documento_id string, titulo string, fecha string, contenido_raw string",
).withColumn("fecha", F.to_date("fecha"))

df_docs.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{NS}.documentos")
assert spark.table(f"{NS}.documentos").count() == 6
display(spark.table(f"{NS}.documentos").select("documento_id", "titulo", "fecha"))

documento_id,titulo,fecha
politica_devoluciones,Política de devoluciones,2026-07-01
contrato_expreso_veloz,Contrato Expreso Veloz,2026-06-15
contrato_paquetes_unidos,Contrato Paquetes Unidos,2026-06-15
ficha_bebidas,Ficha de categoría Bebidas,2026-05-20
ficha_lacteos,Ficha de categoría Lácteos,2026-05-20
proveedor_exoticos,Ficha del proveedor Exóticos Líquidos,2026-04-10


## 2 · Normalización conservadora y offsets

La regla elimina **solo** `guion + un salto de línea` entre letras. Conservamos `contenido_raw`
y un mapa `índice normalizado → índice raw`; así una cita puede volver al original.

In [0]:
CORTE_HIFEN = re.compile(
    r"(?<=[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]{2})-[ \t]*\n[ \t]*(?=[a-záéíóúüñ])"
)

def deshifenar_con_mapa(texto_raw: str):
    salida, mapa = [], []
    cursor = 0
    for match in CORTE_HIFEN.finditer(texto_raw):
        for pos in range(cursor, match.start()):
            salida.append(texto_raw[pos])
            mapa.append(pos)
        cursor = match.end()
    for pos in range(cursor, len(texto_raw)):
        salida.append(texto_raw[pos])
        mapa.append(pos)
    return "".join(salida), mapa

texto_demo, mapa_demo = deshifenar_con_mapa(documentos[0][3])
assert "perecibles" in texto_demo
assert "pere-\ncibles" in documentos[0][3]
assert len(texto_demo) == len(mapa_demo)
print("✅ hifenación controlada: raw intacto, vista normalizada y mapa de offsets")

✅ hifenación controlada: raw intacto, vista normalizada y mapa de offsets


In [0]:
def oraciones_con_offsets(texto: str):
    for match in re.finditer(r"[^.!?]+(?:[.!?]+|$)", texto, flags=re.S):
        segmento = match.group(0)
        izquierda = len(segmento) - len(segmento.lstrip())
        derecha = len(segmento.rstrip())
        inicio = match.start() + izquierda
        fin = match.start() + derecha
        if fin > inicio and len(texto[inicio:fin]) > 20:
            yield inicio, fin, texto[inicio:fin]

filas_chunks = []
for documento_id, titulo, fecha, contenido_raw in documentos:
    normalizado, mapa = deshifenar_con_mapa(contenido_raw)
    spans = list(oraciones_con_offsets(normalizado))
    for chunk_n, (norm_start, norm_end, texto_citable) in enumerate(spans, start=1):
        raw_start = mapa[norm_start]
        raw_end = mapa[norm_end - 1] + 1
        raw_span = contenido_raw[raw_start:raw_end]
        reconstruido, _ = deshifenar_con_mapa(raw_span)
        assert reconstruido.strip() == texto_citable
        previo = spans[chunk_n - 2][2] if chunk_n > 1 else ""
        texto_embedding = (
            f"{titulo}. Contexto anterior: {previo} Fragmento: {texto_citable}"
            if previo
            else f"{titulo}. {texto_citable}"
        )
        chunk_id = hashlib.sha256(f"{documento_id}|{chunk_n}".encode()).hexdigest()
        filas_chunks.append(
            (
                chunk_id,
                documento_id,
                titulo,
                fecha,
                chunk_n,
                raw_start,
                raw_end,
                texto_citable,
                texto_embedding,
            )
        )

chunks = spark.createDataFrame(
    filas_chunks,
    "chunk_id string, documento_id string, titulo string, fecha string, chunk_n int, "
    "raw_start int, raw_end int, texto_citable string, texto_embedding string",
).withColumn("fecha", F.to_date("fecha"))

chunks.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{NS}.chunks")
spark.sql(f"ALTER TABLE {NS}.chunks SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
n_chunks = spark.table(f"{NS}.chunks").count()
assert n_chunks == 17, f"Se esperaban 17 chunks; hay {n_chunks}."

detalle = spark.sql(f"DESCRIBE DETAIL {NS}.chunks").first().asDict()
assert str(detalle["properties"].get("delta.enableChangeDataFeed", "")).lower() == "true"
print(f"✅ {n_chunks} chunks · offsets reversibles · CDF=true")
display(spark.table(f"{NS}.chunks").select("documento_id", "chunk_n", "raw_start", "raw_end", "texto_citable"))

✅ 17 chunks · offsets reversibles · CDF=true


documento_id,chunk_n,raw_start,raw_end,texto_citable
politica_devoluciones,1,0,93,Las devoluciones de alimentos no perecibles se aceptan hasta 30 días después de la entrega.
politica_devoluciones,2,94,214,Productos refrigerados o congelados solo se aceptan dentro de 24 horas si existe evidencia de ruptura de cadena de frío.
politica_devoluciones,3,215,260,El cliente debe informar el número de pedido.
contrato_expreso_veloz,1,0,39,Expreso Veloz cubre Lima Metropolitana.
contrato_expreso_veloz,2,40,95,El compromiso estándar es entregar en 24 horas hábiles.
contrato_expreso_veloz,3,96,198,"Si supera 48 horas, Neptuno puede solicitar una nota de crédito equivalente al 8 por ciento del flete."
contrato_paquetes_unidos,1,0,33,Paquetes Unidos cubre provincias.
contrato_paquetes_unidos,2,34,77,El plazo objetivo es de 3 a 5 días hábiles.
contrato_paquetes_unidos,3,78,210,Mercadería refrigerada requiere embalaje térmico provisto por Neptuno; el transportista registra temperatura en recepción y entrega.
ficha_bebidas,1,0,69,"La categoría Bebidas incluye cafés, tés, cervezas y vinos importados."


## 3 · Embeddings persistidos

Indexamos `texto_embedding`, que puede incorporar contexto anterior. La cita continúa siendo
`texto_citable`, sin overlap, y los offsets siguen apuntando al raw.

In [0]:
chunks_embeddings = (
    spark.table(f"{NS}.chunks")
    .withColumn("embedding", F.expr(f"ai_query('{EMBEDDING_ENDPOINT}', texto_embedding)"))
)
chunks_embeddings.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{NS}.chunks_embeddings"
)
spark.sql(
    f"ALTER TABLE {NS}.chunks_embeddings SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)

dimensiones = {
    r.dims
    for r in spark.table(f"{NS}.chunks_embeddings")
    .select(F.size("embedding").alias("dims"))
    .distinct()
    .collect()
}
assert dimensiones == {1024}
assert spark.table(f"{NS}.chunks_embeddings").count() == n_chunks
print(f"✅ {n_chunks} embeddings persistidos · dimensión 1.024")

✅ 17 embeddings persistidos · dimensión 1.024


## 4 · Tres carriles comparables: BM25, dense y RRF

Los tres devuelven el mismo contrato (`chunk_id`, documento, texto y score). RRF fusiona
posiciones; no reparte cupos fijos entre listas.

In [0]:
STOPWORDS = {
    "a", "al", "con", "cual", "como", "de", "del", "el", "en", "es", "la", "las", "lo",
    "los", "para", "por", "que", "se", "si", "su", "un", "una", "y",
}

def tokenizar(texto: str):
    limpio = unicodedata.normalize("NFKD", texto.lower())
    limpio = "".join(c for c in limpio if not unicodedata.combining(c))
    return [t for t in re.findall(r"[a-z0-9]+", limpio) if len(t) > 1 and t not in STOPWORDS]

chunk_rows = [r.asDict() for r in spark.table(f"{NS}.chunks_embeddings").collect()]
por_id = {r["chunk_id"]: r for r in chunk_rows}
tokens_docs = {r["chunk_id"]: tokenizar(r["texto_embedding"]) for r in chunk_rows}
N = len(tokens_docs)
avgdl = sum(len(ts) for ts in tokens_docs.values()) / N
df_termino = Counter(t for ts in tokens_docs.values() for t in set(ts))

def ranking_bm25(pregunta: str, k: int = TOP_K):
    k1, b = 1.5, 0.75
    consulta = tokenizar(pregunta)
    ranking = []
    for chunk_id, tokens in tokens_docs.items():
        frecuencias = Counter(tokens)
        score = 0.0
        for termino in consulta:
            if termino not in frecuencias:
                continue
            idf = math.log(1 + (N - df_termino[termino] + 0.5) / (df_termino[termino] + 0.5))
            tf = frecuencias[termino]
            score += idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * len(tokens) / avgdl))
        if score > 0:
            ranking.append((chunk_id, score))
    return sorted(ranking, key=lambda x: (-x[1], x[0]))[:k]

def embedding_consulta(pregunta: str):
    return (
        spark.createDataFrame([(pregunta,)], "texto string")
        .select(F.expr(f"ai_query('{EMBEDDING_ENDPOINT}', texto)").alias("embedding"))
        .first()["embedding"]
    )

def ranking_dense(pregunta: str, k: int = TOP_K):
    qvec = embedding_consulta(pregunta)
    ranking = [(r["chunk_id"], cosine(qvec, r["embedding"])) for r in chunk_rows]
    return sorted(ranking, key=lambda x: (-x[1], x[0]))[:k]

def ranking_rrf(lexico, dense, k: int = TOP_K, constante: int = 60):
    scores = defaultdict(float)
    for lista in (lexico, dense):
        for posicion, (chunk_id, _) in enumerate(lista, start=1):
            scores[chunk_id] += 1 / (constante + posicion)
    return sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]

def recuperar(pregunta: str, k: int = TOP_K):
    lexico = ranking_bm25(pregunta, k)
    dense = ranking_dense(pregunta, k)
    hibrido = ranking_rrf(lexico, dense, k)
    return {"bm25": lexico, "dense": dense, "rrf": hibrido}

pregunta_demo = "¿En cuánto tiempo puedo retornar un congelado si se rompió la cadena de frío?"
carriles_demo = recuperar(pregunta_demo)
for nombre, ranking in carriles_demo.items():
    primero = por_id[ranking[0][0]]
    print(f"{nombre:>5} → {primero['titulo']} · score={ranking[0][1]:.4f}")

 bm25 → Política de devoluciones · score=3.4918
dense → Política de devoluciones · score=0.7002
  rrf → Política de devoluciones · score=0.0328


### ⭐ Demo · inspeccionar lo recuperado antes de generar

In [0]:
filas_demo = []
for carril, ranking in carriles_demo.items():
    for posicion, (chunk_id, score) in enumerate(ranking, start=1):
        r = por_id[chunk_id]
        filas_demo.append((carril, posicion, score, r["documento_id"], r["titulo"], r["texto_citable"]))

display(
    spark.createDataFrame(
        filas_demo,
        "carril string, posicion int, score double, documento_id string, titulo string, texto_citable string",
    ).orderBy("carril", "posicion")
)

carril,posicion,score,documento_id,titulo,texto_citable
bm25,1,3.491807068795499,politica_devoluciones,Política de devoluciones,El cliente debe informar el número de pedido.
bm25,2,3.1411135664154863,politica_devoluciones,Política de devoluciones,Productos refrigerados o congelados solo se aceptan dentro de 24 horas si existe evidencia de ruptura de cadena de frío.
dense,1,0.7001836648194464,politica_devoluciones,Política de devoluciones,El cliente debe informar el número de pedido.
dense,2,0.6263514017720835,politica_devoluciones,Política de devoluciones,Productos refrigerados o congelados solo se aceptan dentro de 24 horas si existe evidencia de ruptura de cadena de frío.
dense,3,0.4913537673304683,politica_devoluciones,Política de devoluciones,Las devoluciones de alimentos no perecibles se aceptan hasta 30 días después de la entrega.
dense,4,0.4184449387858501,contrato_paquetes_unidos,Contrato Paquetes Unidos,Mercadería refrigerada requiere embalaje térmico provisto por Neptuno; el transportista registra temperatura en recepción y entrega.
rrf,1,0.03278688524590164,politica_devoluciones,Política de devoluciones,El cliente debe informar el número de pedido.
rrf,2,0.03225806451612903,politica_devoluciones,Política de devoluciones,Productos refrigerados o congelados solo se aceptan dentro de 24 horas si existe evidencia de ruptura de cadena de frío.
rrf,3,0.015873015873015872,politica_devoluciones,Política de devoluciones,Las devoluciones de alimentos no perecibles se aceptan hasta 30 días después de la entrega.
rrf,4,0.015625,contrato_paquetes_unidos,Contrato Paquetes Unidos,Mercadería refrigerada requiere embalaje térmico provisto por Neptuno; el transportista registra temperatura en recepción y entrega.


## 5 · Gold set: retrieval y abstención

Cuatro preguntas positivas y una negativa. La negativa no exige que el buscador devuelva cero
vecinos — siempre habrá un vecino geométrico—; exige que el gate reconozca evidencia insuficiente.

In [0]:
gold = [
    ("devolucion_frio", "¿Cuánto tiempo tengo para retornar un congelado si se rompió la cadena de frío?", "politica_devoluciones", "24 horas"),
    ("compensacion_flete", "¿Qué compensación corresponde si Expreso Veloz demora más de dos días?", "contrato_expreso_veloz", "8 por ciento"),
    ("lacteo_fuera_rango", "¿Qué ocurre con un lote lácteo recibido a 8 grados?", "ficha_lacteos", "cuarentena"),
    ("lead_time", "¿Cuál es el tiempo de abastecimiento de Exóticos Líquidos?", "proveedor_exoticos", "18 días"),
    ("sin_evidencia", "¿Cuál es la contraseña del WiFi del almacén?", None, None),
]

evaluacion = []
cache_carriles = {}
for caso, pregunta, documento_esperado, evidencia_esperada in gold:
    carriles = recuperar(pregunta)
    cache_carriles[caso] = carriles
    bm25_top = por_id[carriles["bm25"][0][0]]["documento_id"] if carriles["bm25"] else None
    dense_top = por_id[carriles["dense"][0][0]]["documento_id"]
    rrf_top = por_id[carriles["rrf"][0][0]]["documento_id"]
    lex_score = carriles["bm25"][0][1] if carriles["bm25"] else 0.0
    dense_score = carriles["dense"][0][1]
    evidencia_suficiente = lex_score > 0 or dense_score >= 0.65
    docs_rrf = [por_id[chunk_id]["documento_id"] for chunk_id, _ in carriles["rrf"]]
    hit_top3 = documento_esperado in docs_rrf if documento_esperado else None
    abstiene = not evidencia_suficiente
    evaluacion.append(
        (
            caso,
            pregunta,
            documento_esperado,
            evidencia_esperada,
            bm25_top,
            dense_top,
            rrf_top,
            float(lex_score),
            float(dense_score),
            hit_top3,
            abstiene,
        )
    )

df_eval = spark.createDataFrame(
    evaluacion,
    "caso string, pregunta string, documento_esperado string, evidencia_esperada string, "
    "bm25_top1 string, dense_top1 string, rrf_top1 string, bm25_score double, dense_score double, "
    "rrf_hit_top3 boolean, abstiene boolean",
)
display(df_eval)

positivos = df_eval.filter("documento_esperado IS NOT NULL")
assert positivos.filter("NOT rrf_hit_top3").count() == 0, "RRF perdió evidencia del gold set en top-3."
assert positivos.filter("rrf_top1 = documento_esperado").count() >= 3, "RRF debe acertar al menos 3/4 en top-1."
assert df_eval.filter("caso = 'sin_evidencia' AND abstiene").count() == 1, "El gate no abstuvo el caso negativo."

df_eval.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{NS}.evaluacion_retrieval")
print("✅ gold set: 4 positivos con evidencia en top-3 + 1 negativo con abstención")

caso,pregunta,documento_esperado,evidencia_esperada,bm25_top1,dense_top1,rrf_top1,bm25_score,dense_score,rrf_hit_top3,abstiene
devolucion_frio,¿Cuánto tiempo tengo para retornar un congelado si se rompió la cadena de frío?,politica_devoluciones,24 horas,politica_devoluciones,politica_devoluciones,politica_devoluciones,3.491807068795499,0.7100092113462734,true,false
compensacion_flete,¿Qué compensación corresponde si Expreso Veloz demora más de dos días?,contrato_expreso_veloz,8 por ciento,contrato_expreso_veloz,contrato_expreso_veloz,contrato_expreso_veloz,5.686239371877282,0.5459187619567855,true,false
lacteo_fuera_rango,¿Qué ocurre con un lote lácteo recibido a 8 grados?,ficha_lacteos,cuarentena,ficha_lacteos,ficha_lacteos,ficha_lacteos,3.3489619023699957,0.7138951136055246,true,false
lead_time,¿Cuál es el tiempo de abastecimiento de Exóticos Líquidos?,proveedor_exoticos,18 días,proveedor_exoticos,proveedor_exoticos,proveedor_exoticos,5.227925333211551,0.7120559208780757,true,false
sin_evidencia,¿Cuál es la contraseña del WiFi del almacén?,null,null,null,ficha_bebidas,ficha_bebidas,0.0,0.27765123896035926,null,true


✅ gold set: 4 positivos con evidencia en top-3 + 1 negativo con abstención


In [0]:
fallos_top3 = []
for caso, _, documento_esperado, _ in gold:
    if documento_esperado is None:
        continue
    docs_top3 = [
        por_id[chunk_id]["documento_id"]
        for chunk_id, _ in cache_carriles[caso]["rrf"][:3]
    ]
    if documento_esperado not in docs_top3:
        fallos_top3.append((caso, documento_esperado, docs_top3))

assert not fallos_top3, f"Casos sin evidencia RRF en top-3: {fallos_top3}"
print("✅ Verificación estricta: 4/4 positivos con evidencia RRF en top-3")

✅ Verificación estricta: 4/4 positivos con evidencia RRF en top-3


In [0]:
for ep in w.serving_endpoints.list():
      print(
          ep.name,
          "| tarea:", getattr(ep, "task", None),
          "| estado:", getattr(getattr(ep, "state", None), "ready", None),
      )

databricks-gpt-oss-120b | tarea: llm/v1/chat | estado: EndpointStateReady.READY
databricks-gpt-oss-20b | tarea: llm/v1/chat | estado: EndpointStateReady.READY
databricks-qwen3-next-80b-a3b-instruct | tarea: llm/v1/chat | estado: EndpointStateReady.READY
databricks-qwen35-122b-a10b | tarea: llm/v1/chat | estado: EndpointStateReady.READY
databricks-llama-4-maverick | tarea: llm/v1/chat | estado: EndpointStateReady.READY
databricks-gemma-3-12b | tarea: llm/v1/chat | estado: EndpointStateReady.READY
databricks-gte-large-en | tarea: llm/v1/embeddings | estado: EndpointStateReady.READY
databricks-bge-large-en | tarea: llm/v1/embeddings | estado: EndpointStateReady.READY
databricks-meta-llama-3-1-8b-instruct | tarea: llm/v1/chat | estado: EndpointStateReady.READY
databricks-meta-llama-3-3-70b-instruct | tarea: llm/v1/chat | estado: EndpointStateReady.READY
databricks-qwen3-embedding-0-6b | tarea: llm/v1/embeddings | estado: EndpointStateReady.READY


## 6 · Reranking listwise sobre el shortlist

El reranker **no reemplaza** el primer retrieval: relee juntos los candidatos y devuelve un
nuevo orden. Medimos también si produjo una salida válida; un ranking con empates o IDs inventados
no sirve para un gate de confianza.

In [0]:
candidatos = cache_carriles["devolucion_frio"]["rrf"]
lineas = []
for chunk_id, _ in candidatos:
    r = por_id[chunk_id]
    lineas.append(f"ID={chunk_id}\nTÍTULO={r['titulo']}\nTEXTO={r['texto_citable']}")

prompt_rerank = f"""
Ordena TODOS los candidatos del más al menos relevante para responder la pregunta.
Pregunta: {gold[0][1]}

{chr(10).join(lineas)}

Devuelve únicamente un arreglo JSON de IDs, sin markdown ni explicación.
No agregues, repitas ni elimines IDs.
""".strip()

salida_rerank = (
    spark.createDataFrame([(prompt_rerank,)], "prompt string")
    .select(F.expr(f"ai_query('{MODELO}', prompt)").alias("respuesta"))
    .first()["respuesta"]
)

match = re.search(r"\[[\s\S]*\]", salida_rerank)
ids_rerank = json.loads(match.group(0)) if match else []
ids_candidatos = [chunk_id for chunk_id, _ in candidatos]
rerank_valido = (
    len(ids_rerank) == len(ids_candidatos)
    and len(set(ids_rerank)) == len(ids_rerank)
    and set(ids_rerank) == set(ids_candidatos)
)
assert rerank_valido, f"El reranker no devolvió una permutación válida: {salida_rerank}"
print("✅ reranker listwise: permutación válida del shortlist")
for posicion, chunk_id in enumerate(ids_rerank, start=1):
    print(f"{posicion}. {por_id[chunk_id]['titulo']} — {por_id[chunk_id]['texto_citable']}")

✅ reranker listwise: permutación válida del shortlist
1. Política de devoluciones — Productos refrigerados o congelados solo se aceptan dentro de 24 horas si existe evidencia de ruptura de cadena de frío.
2. Contrato Paquetes Unidos — Mercadería refrigerada requiere embalaje térmico provisto por Neptuno; el transportista registra temperatura en recepción y entrega.
3. Política de devoluciones — El cliente debe informar el número de pedido.
4. Política de devoluciones — Las devoluciones de alimentos no perecibles se aceptan hasta 30 días después de la entrega.


## 7 · Generación grounded con cita

Para la respuesta final agregamos los tres primeros spans citables. El modelo no ve el texto
usado para embeddings ni puede inventar una fuente fuera del bloque de evidencias.

In [0]:
top_contexto = ids_rerank[:3]
bloques = []
for chunk_id in top_contexto:
    r = por_id[chunk_id]
    bloques.append(f"[{r['titulo']}] {r['texto_citable']}")

prompt_grounded = f"""
Responde en español usando EXCLUSIVAMENTE estas evidencias:

{chr(10).join(bloques)}

Pregunta: {gold[0][1]}

Reglas:
- Si la evidencia no alcanza, responde exactamente: No hay evidencia suficiente.
- Cita cada afirmación factual con el título entre corchetes.
- Máximo 90 palabras.
""".strip()

respuesta = (
    spark.createDataFrame([(prompt_grounded,)], "prompt string")
    .select(F.expr(f"ai_query('{MODELO}', prompt)").alias("respuesta"))
    .first()["respuesta"]
)
print(respuesta)
assert "24" in respuesta
assert "[Política de devoluciones]" in respuesta
assert "No hay evidencia suficiente" not in respuesta
print("✅ respuesta positiva: dato esperado + cita válida")

Tienes 24 horas para retornar el producto congelado, siempre que cuentes con evidencia de la ruptura de la cadena de frío. [Política de devoluciones]
✅ respuesta positiva: dato esperado + cita válida


### ⭐ Caso negativo · abstener antes de llamar al LLM

In [0]:
negativo = df_eval.filter("caso = 'sin_evidencia'").first().asDict()
respuesta_negativa = (
    "No hay evidencia suficiente."
    if negativo["abstiene"]
    else "ERROR: el gate permitió generar sin evidencia."
)
print(respuesta_negativa)
assert respuesta_negativa == "No hay evidencia suficiente."

No hay evidencia suficiente.


## 8 · Ruta administrada con Databricks AI Search

Este workspace no necesita un índice para demostrar embeddings. Para escalar el mismo contrato:

1. crea un endpoint **standard** de AI Search;
2. crea un índice **Delta Sync** sobre `<catalogo>.<schema>.chunks_embeddings`;
3. usa `chunk_id` como PK y la columna `embedding` existente (1.024 dims), o deja que el índice
   llame al endpoint administrado;
4. compara `ANN`, `FULL_TEXT` y `HYBRID` con el mismo gold set;
5. conserva filtros de vigencia, región y permisos antes de generar.

> CDF ya quedó habilitado en las tablas fuente del laboratorio.

In [0]:
resumen = {
    "namespace": NS,
    "documentos": spark.table(f"{NS}.documentos").count(),
    "chunks": spark.table(f"{NS}.chunks").count(),
    "embedding_endpoint": EMBEDDING_ENDPOINT,
    "embedding_dims": next(iter(dimensiones)),
    "gold_positivos_top3": positivos.filter("rrf_hit_top3").count(),
    "gold_negativos_abstenidos": df_eval.filter("caso = 'sin_evidencia' AND abstiene").count(),
    "rerank_valido": rerank_valido,
    "respuesta_citada": "[Política de devoluciones]" in respuesta,
}
display(spark.createDataFrame([(k, str(v)) for k, v in resumen.items()], "control string, valor string"))
assert resumen == {
    "namespace": NS,
    "documentos": 6,
    "chunks": 17,
    "embedding_endpoint": EMBEDDING_ENDPOINT,
    "embedding_dims": 1024,
    "gold_positivos_top3": 4,
    "gold_negativos_abstenidos": 1,
    "rerank_valido": True,
    "respuesta_citada": True,
}
print("🎓 S04 LISTA: corpus + offsets + BM25/dense/RRF + rerank + cita + abstención + evaluación persistida")

control,valor
namespace,neptuno_emerson_suarez.rag
documentos,6
chunks,17
embedding_endpoint,databricks-qwen3-embedding-0-6b
embedding_dims,1024
gold_positivos_top3,4
gold_negativos_abstenidos,1
rerank_valido,True
respuesta_citada,True


🎓 S04 LISTA: corpus + offsets + BM25/dense/RRF + rerank + cita + abstención + evaluación persistida


### Verificación final de las tablas

In [0]:
tablas_esperadas = {
    "documentos": 6,
    "chunks": 17,
    "chunks_embeddings": 17,
    "evaluacion_retrieval": 5,
}

for nombre, filas_esperadas in tablas_esperadas.items():
    tabla = f"{NS}.{nombre}"
    assert spark.catalog.tableExists(tabla), f"Falta {tabla}"
    filas = spark.table(tabla).count()
    assert filas == filas_esperadas, f"{tabla}: se esperaban {filas_esperadas} filas y hay {filas}"

assert spark.table(f"{NS}.chunks").filter(
    "raw_start IS NULL OR raw_end IS NULL OR texto_citable IS NULL"
).count() == 0

assert spark.table(f"{NS}.chunks_embeddings").filter(
    "embedding IS NULL OR size(embedding) <> 1024"
).count() == 0

display(spark.table(f"{NS}.evaluacion_retrieval").orderBy("caso"))
print("🎉 Entregable S04: cuatro tablas completas y evaluación inspeccionable")

caso,pregunta,documento_esperado,evidencia_esperada,bm25_top1,dense_top1,rrf_top1,bm25_score,dense_score,rrf_hit_top3,abstiene
compensacion_flete,¿Qué compensación corresponde si Expreso Veloz demora más de dos días?,contrato_expreso_veloz,8 por ciento,contrato_expreso_veloz,contrato_expreso_veloz,contrato_expreso_veloz,5.686239371877282,0.5459187619567855,true,false
devolucion_frio,¿Cuánto tiempo tengo para retornar un congelado si se rompió la cadena de frío?,politica_devoluciones,24 horas,politica_devoluciones,politica_devoluciones,politica_devoluciones,3.491807068795499,0.7100092113462734,true,false
lacteo_fuera_rango,¿Qué ocurre con un lote lácteo recibido a 8 grados?,ficha_lacteos,cuarentena,ficha_lacteos,ficha_lacteos,ficha_lacteos,3.3489619023699957,0.7138951136055246,true,false
lead_time,¿Cuál es el tiempo de abastecimiento de Exóticos Líquidos?,proveedor_exoticos,18 días,proveedor_exoticos,proveedor_exoticos,proveedor_exoticos,5.227925333211551,0.7120559208780757,true,false
sin_evidencia,¿Cuál es la contraseña del WiFi del almacén?,null,null,null,ficha_bebidas,ficha_bebidas,0.0,0.27765123896035926,null,true


🎉 Entregable S04: cuatro tablas completas y evaluación inspeccionable
